In [37]:
import ROOT
import numpy as np
import pandas as pd
from array import array

In [49]:
CHANNEL = "Skim1E2Mu"
ERA = "2017"
MASSPOINT = "MHc-160_MA-120"
BKGs = ["nonprompt", "conversion", "WZ", "ZZ", "ttW", "ttZ", "ttH",
        "GluGluHToZZTo4L", "VBF_HToZZTo4L", "WWW", "WWZ", "WZZ", "ZZZ", "tZq", "tHq", "TTTT"]

In [50]:
WORKDIR = "/home/choij/workspace/ChargedHiggsAnalysisV2/SignalRegionStudy"
SAMPLEDIR = f"../samples/{ERA}/{CHANNEL}/{MASSPOINT}"

# Get fitting results
result = ROOT.TFile.Open(f"{WORKDIR}/templates/{ERA}/{CHANNEL.replace('Skim', 'SR')}/{MASSPOINT}/Shape/Baseline/fit_result.root").Get("fitresult_model_data")
mA = result.floatParsFinal().find("mA").getVal()
sigma = result.floatParsFinal().find("sigma").getVal()
width = result.floatParsFinal().find("width").getVal()
window = np.sqrt(sigma**2 + width**2)

h = ROOT.TH1D("h", "h", 100, mA-5*window, mA+5*window)
f = ROOT.TFile.Open(f"{SAMPLEDIR}/{MASSPOINT}.root")
tree = f.Get(f"{MASSPOINT}_Central")
for evt in tree:
    h.Fill(evt.mass, evt.weight)
f.Close()
err = array('d', [0])
integral = h.IntegralAndError(h.FindBin(mA-5*window), h.FindBin(mA+5*window), err)
print(integral, err[0])

15.968540265695907 0.07069903891636156


In [51]:
for bkg in BKGs:
    f = ROOT.TFile.Open(f"{SAMPLEDIR}/{bkg}.root")
    h = ROOT.TH1D("h", "h", 100, mA-5*window, mA+5*window); h.SetDirectory(0)
    tree = f.Get(f"{bkg}_Central")
    if not tree:
        tree = f.Get("others_Central")
    for evt in tree:
        h.Fill(evt.mass, evt.weight)
    f.Close()
    err = array('d', [0])
    integral = h.IntegralAndError(h.FindBin(mA-5*window), h.FindBin(mA+5*window), err)
    print(bkg, integral, err[0])

nonprompt 39.1848138937345 4.463161746158609
conversion 1.647315558392798 0.6432480211018707
WZ 2.14396156078187 0.34127031318458295
ZZ 0.5853993560824031 0.019839945054483835
ttW 7.7073336859692745 0.16228142195877956
ttZ 3.781781468006537 0.09932865504534513
ttH 2.794061562254899 0.06691641954937545
GluGluHToZZTo4L 0.0 0.0
VBF_HToZZTo4L 7.722477788655091e-05 7.722477788655091e-05
WWW 0.08873864247949509 0.009952146770379061
WWZ -0.0330056439499075 0.0330056439499075
WZZ 0.012154768138010075 0.0017975930161901689
ZZZ 0.0002106665935757661 0.00012272950835383643
tZq 0.38734426384421383 0.042823367154948375
tHq 0.721401977416761 0.02579382128659194
TTTT 0.47200981012222953 0.008552543962659191
